# Phase 3i: control run of the main finding on the export of 2026-09-30

Plan: `plans/phase_3i.md` (written and committed before computation, one amendment after). Computation: `analysis/phase3i_check.py`; outputs in `data/phase3i/`. Descriptive: no test, no register entry. Window: the seven UBS earnings calls from 2023-Q2 to 2024-Q4.

In [1]:
from pathlib import Path
import json
import pandas as pd

ROOT = Path.cwd().parents[1] if Path.cwd().name == "roman" else Path.cwd()
OUT = ROOT / "notebooks" / "roman" / "data" / "phase3i"
pd.set_option("display.max_colwidth", 200)
summary = json.loads((OUT / "summary.json").read_text())

## Exports

In [2]:
pd.DataFrame(summary["exports"]).T

,old_2026-09-23,new_2026-09-30,equal_to_final_folder
all_sentences.csv,a6c15d3c4430bf855a961a8a151bf08cf7a4f9517454c27c133756dd3ab64b57,1b205c449b8131d882a9ba681ca86ddde2337d5c73033e36e468f656d7f5cc4b,True
all_utterances.csv,e268194163d53421e0e4efe1b173c90b566b3931252c18786f863e92f70fa190,6fe0457bdfa3a7f6d1f140f3f0c17e571d2340dd424260ccec046f69050af121,True


The two exports are different files, and the new one is byte-identical to the copy in the submission folder.

## Sentences and utterances per call

In [3]:
c = pd.read_csv(OUT / "call_counts.csv")
c[c.section == "all"].pivot_table(index="quarter", columns="unit", values=["old", "new", "difference"])

difference                  new                  old           
unit     sentences utterances sentences utterances sentences utterances
quarter                                                                
2023-Q2        0.0        0.0     672.0       45.0     672.0       45.0
2023-Q3        0.0        0.0     534.0       40.0     534.0       40.0
2023-Q4        0.0        0.0     804.0       61.0     804.0       61.0
2024-Q1        0.0        0.0     501.0       50.0     501.0       50.0
2024-Q2        0.0        0.0     447.0       42.0     447.0       42.0
2024-Q3        0.0        0.0     508.0       36.0     508.0       36.0
2024-Q4        0.0        0.0     709.0       60.0     709.0       60.0

In [4]:
print("rows by section and speaker role with a difference:", int((c.difference != 0).sum()), "of", len(c))

rows by section and speaker role with a difference: 0 of 58


Every call has the same number of sentences and utterances in both exports, in total and by section and speaker role (4,175 sentences and 334 utterances in the window).

## Sentence comparison

In [5]:
d = pd.read_csv(OUT / "sentence_diff.csv")
print("sentences old, new, same:", summary["sentences"]["old"], summary["sentences"]["new"], summary["sentences"]["same"])
print("sentences that are not the same in both exports:", len(d))

sentences old, new, same: 4175 4175 4175
sentences that are not the same in both exports: 0


All 4,175 sentences of the window occur in both exports with the same normalised text, speaker, role and section. There is nothing to explain by sentence splitting, filler sentences or speaker assignment.

## Source statements of the 39 targets

In [6]:
m = pd.read_csv(OUT / "units_match.csv", keep_default_na=False)
m[["unit", "phase", "call", "found", "same_wording", "same_speaker", "same_figures", "sentence_id_old", "sentence_id_new", "position_in_call_new", "sentence_number_new"]]

,unit,phase,call,found,same_wording,same_speaker,same_figures,sentence_id_old,sentence_id_new,position_in_call_new,sentence_number_new
0,T16,3d,UBS 2023-Q3 earnings,True,True,True,True,16014,25533,2,139
1,T25,3d,UBS 2023-Q4 earnings,True,True,True,True,16491,26010,3,41
2,T27,3d,UBS 2023-Q4 earnings,True,True,True,True,16542,26061,3,92
3,T32,3d,UBS 2023-Q4 earnings,True,True,True,True,16639,26158,4,82
4,T33,3d,UBS 2023-Q4 earnings,True,True,True,True,16641,26160,4,84
5,T68,3d,UBS 2024-Q4 earnings,True,True,True,True,18959,28478,4,151
6,T10,3d,UBS 2023-Q2 earnings,True,True,True,True,15272,24791,1,140
7,T12,3d,UBS 2023-Q2 earnings,True,True,True,True,15764,25283,40,1
8,T13,3d,UBS 2023-Q2 earnings,True,True,True,True,15765,25284,40,2
9,T9,3d,UBS 2023-Q2 earnings,True,True,True,True,15264,24783,1,132


In [7]:
m[["found", "same_wording", "same_speaker", "same_figures"]].sum().to_frame("units of 39")

,units of 39
found,39
same_wording,39
same_speaker,39
same_figures,39


All 39 source statements are found in the new export in the same call, with the same speaker, figures and wording. Only `sentence_id` differs, as expected: it is stable only within one release.

## Rule candidates and the set S

In [8]:
pd.DataFrame({k: summary[k] for k in ("rule_candidates", "set_s")}).T

,old,new,same
rule_candidates,105,105,105
set_s,68,68,68


In [9]:
pd.Series(summary["finbert_label_agreement"]).to_frame("statements")

,statements
paired_statements,3177
same_label,3177
different_label,0


The phase 3 rules select the same 105 candidates and FinBERT-FLS gives the same label to all 3,177 management statements, so S has the same 68 UBS sentences. Model, revision, maximum length, batch size and seed are those of phase 3g.

In [10]:
pd.Series(summary["model"]).to_frame("value")

,value
model,yiyanghkust/finbert-fls
revision,443586dc31c765c0aaf1c4daaed8cf3643c92fa5
device,cuda
seed,0
max_length,256
batch_size,64


## Statements for reading

In [11]:
r = pd.read_csv(OUT / "only_new_for_reading.csv")
g = pd.read_csv(OUT / "only_old.csv")
print("only new, for Roman to read:", len(r))
print("only old:", len(g))

only new, for Roman to read: 0
only old: 0


No statement is new in the rule candidates or in S, and none dropped out. There is nothing to read.

## Additional exact comparison (amendment, after the result)

The plan compares normalised text. This cell compares the raw files on the stable sentence key, without normalisation, and the utterance text.

In [12]:
KEY = ["bank", "quarter", "call_type", "position_in_call", "sentence_number"]
OLD = ROOT / "data" / "results" / "2026-09-23"
NEW = Path.home() / "projects" / "earnings-call-nlp-pipeline" / "exports" / "2026-09-30"
def window(x):
    return x[(x.bank == "UBS") & (x.call_type == "earnings") & (x.quarter >= "2023-Q2") & (x.quarter <= "2024-Q4")]
so, sn = pd.read_csv(OLD / "all_sentences.csv"), pd.read_csv(NEW / "all_sentences.csv")
j = window(so).merge(window(sn), on=KEY, how="outer", suffixes=("_old", "_new"), indicator=True)
rows = {"sentence keys in both exports": (j._merge == "both").sum()}
for col in ("sentence", "speaker_name", "speaker_role", "section", "call_date", "sentence_id"):
    rows[f"{col} identical"] = (j[col + "_old"].astype(str) == j[col + "_new"].astype(str)).sum()
uo, un = window(pd.read_csv(OLD / "all_utterances.csv")), window(pd.read_csv(NEW / "all_utterances.csv"))
ju = uo.merge(un, on=KEY[:4], how="outer", suffixes=("_old", "_new"), indicator=True)
rows["utterance keys in both exports"] = (ju._merge == "both").sum()
rows["utterance text identical"] = (ju.text_old == ju.text_new).sum()
pd.Series(rows).to_frame(f"of {len(j)} sentences / {len(ju)} utterances")

,of 4175 sentences / 334 utterances
sentence keys in both exports,4175
sentence identical,4175
speaker_name identical,4175
speaker_role identical,4175
section identical,4175
call_date identical,4175
sentence_id identical,0
utterance keys in both exports,334
utterance text identical,334


In the window the two exports are identical character for character on the same keys; only `sentence_id` was renumbered.

In [13]:
co = so.groupby(["bank", "quarter", "call_type"]).size().rename("old")
cn = sn.groupby(["bank", "quarter", "call_type"]).size().rename("new")
both = pd.concat([co, cn], axis=1).dropna().astype(int)
both["difference"] = both.new - both.old
both[both.difference != 0]

old  new  difference
bank quarter call_type                      
JPM  2023-Q1 earnings   557  552          -5
     2023-Q2 earnings   581  578          -3
     2023-Q3 earnings   569  564          -5
     2023-Q4 earnings   462  457          -5
     2024-Q1 earnings   641  637          -4
     2024-Q2 earnings   505  502          -3
     2024-Q3 earnings   665  657          -8
     2024-Q4 earnings   506  502          -4

Among the calls that both exports contain, the sentence counts differ only for the eight JPMorgan earnings calls (3 to 8 sentences fewer each). These lie outside the window and outside the main finding; all eight UBS calls and the JPMorgan event call have the same counts.

## Result against the decision rule

Condition (a) is met: all 39 source statements are present with the same call, speaker, figure and wording. Condition (b) is met: the candidate search yields no new candidate, so nothing awaits reading. The finding **holds** on the export of 2026-09-30, and the count of 9 call-only figures, with T30 separate, is unchanged.

## Limits

The result concerns the call side only: the classes A, B and C rest on the reports and were not re-read. It covers the UBS calls of the main finding; the JPMorgan earnings calls changed between the exports, which matters for phases 1 and 2 and was not examined here. The calls of 2022 and 2025 in the new export were not searched for targets.